# Monitoring infrastructure drift with Prometheus and Terraform

This notebook walks through a practical workflow for detecting when live infrastructure diverges from its Terraform-defined desired state. It uses Prometheus metrics as the live-signal layer and a Python script to compare those metrics against the Terraform state snapshot.

## What this covers

- Exporting Terraform state as JSON for programmatic access.
- Querying Prometheus for live resource metrics (instance count, CPU, memory).
- Comparing expected vs. actual resource counts.
- Visualizing drift magnitude over time.

In [ ]:
# --- Terraform setup ---
# A minimal Terraform configuration that provisions two AWS EC2 instances.
# The state file this produces is the source of truth for the comparison.

terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  region = "us-east-1"
}

resource "aws_instance" "app" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.micro"
  count         = 2

  tags = {
    Name = "app-server"
  }
}

output "instance_ids" {
  value = aws_instance.app[*].id
}

In [ ]:
# --- Export Terraform state to JSON ---
# Run this from the directory containing the Terraform configuration above.

import json
import subprocess

def export_terraform_state():
    """
    Capture the Terraform state as JSON so we can inspect expected resources.
    Falls back to a local state file if 'terraform show' is unavailable.
    """
    try:
        result = subprocess.run(
            ["terraform", "show", "-json"],
            capture_output=True,
            text=True,
            check=True,
        )
        state = json.loads(result.stdout)
    except (subprocess.CalledProcessError, FileNotFoundError):
        # Fallback: read a local state file for demonstration
        with open("terraform.tfstate", "r") as f:
            state = json.load(f)

    resources = state.get("values", {}).get("root_module", {}).get("resources", [])
    instances = [
        r for r in resources if r.get("type") == "aws_instance"
    ]
    return instances


instances = export_terraform_state()
print(f"Expected instances per Terraform state: {len(instances)}")
for inst in instances:
    print(f"  - {inst.get('name')}: {inst.get('values', {}).get('id', 'n/a')}")

In [ ]:
# --- Query Prometheus for live instance metrics ---
# Prometheus must be scraping the AWS EC2 instances for these queries to return data.

import requests

PROMETHEUS_URL = "http://localhost:9090"


def query_prometheus(promql: str) -> float:
    """Run an instant PromQL query and return the first result value."""
    resp = requests.get(
        f"{PROMETHEUS_URL}/api/v1/query",
        params={"query": promql},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json().get("data", {}).get("result", [])
    if not results:
        return 0.0
    return float(results[0]["value"][1])


# Count how many EC2 instances with Name=app-server are currently up.
live_count = query_prometheus(
    'count(up{job="aws_ec2", Name="app-server"} == 1)'
)
print(f"Live instances reporting to Prometheus: {live_count:.0f}")

In [ ]:
# --- Drift detection ---

expected = len(instances)
actual   = int(live_count)
drift    = actual - expected

if drift == 0:
    status = "No drift — live count matches Terraform state."
elif drift > 0:
    status = (
        f"Drift detected: {actual} live instances but Terraform expects {expected}. "
        f"Extra instances may be unmanaged."
    )
else:
    status = (
        f"Drift detected: only {actual} live instances but Terraform expects {expected}. "
        f"Instances may have been deleted outside Terraform."
    )

print(status)

## Visualizing drift over time

The cell below uses Matplotlib to plot expected vs. live instance counts. Run this after collecting historical snapshots from Terraform state exports and Prometheus queries to see drift trends.

In [ ]:
import matplotlib.pyplot as plt

# Example historical snapshots (replace with real data collection loop).
timestamps   = ["T0", "T1", "T2", "T3", "T4"]
expected_ts  = [2, 2, 2, 2, 2]
actual_ts    = [2, 2, 1, 2, 3]  # drift events at T2 and T4

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timestamps, expected_ts, label="Expected (Terraform state)", marker="o")
ax.plot(timestamps, actual_ts,   label="Live (Prometheus)",             marker="s")
ax.set_title("Infrastructure drift: expected vs. live instance count")
ax.set_xlabel("Snapshot")
ax.set_ylabel("Instance count")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Key takeaways

- **Terraform state is the source of truth** for what should exist; export it regularly with `terraform show -json` for programmatic comparison.
- **Prometheus provides the live signal** — the gap between expected and actual counts is the drift metric.
- **Automate the comparison** in a scheduled job or CI pipeline so drift surfaces before it becomes an incident.
- **Visualize trends** rather than single snapshots; a gradual count decline is easier to attribute to a specific change than a sudden drop.